# Prerequisites

In [ ]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8
# French original (used in part 4)
!wget -nc -O le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Define the rdd
# (relative path: works in Colab, where the working dir is /content, and locally)
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

**Explanation**

The output is not a list of words but a description of an RDD object, something like
`PythonRDD[2] at RDD at PythonRDD.scala:53`.

This is because `flatMap` is a **transformation**, and Spark transformations are **lazy**:
nothing is read or computed at this point. Spark only records the *recipe*
(the lineage: read the file → split each line on spaces → flatten) in a DAG.
`words` is therefore just a reference to a distributed dataset that *will* be computed
later, when an action is called. The number between brackets is the RDD id in the
SparkContext.

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

**Explanation**

`collect()` is an **action**: it forces Spark to actually execute the whole lineage
(read the file, split every line, flatten the result) and to bring **all** the elements
back from the executors to the driver as a regular Python `list`.

Differences with the previous cell:
- previous cell → a lazy RDD object, no job launched, nothing in memory;
- this cell → a Spark job is triggered (visible in the Spark UI) and we get the real
  words, one element per word.

We can also see that splitting on `' '` is naive: there are empty strings `''`
(blank lines and consecutive spaces), capitalised words, and words glued to punctuation
(`"Fogg,"`, `"said."`...). These issues are fixed in steps a–f.

Warning: `collect()` loads the whole dataset into the driver's memory. It is fine for a
book, but on real big data we should prefer `take(n)`, `count()` or writing to storage.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

In [ ]:
%%time
# map: ONE output element per line -> an RDD of lists
lines_as_lists = rdd.map(lambda line: line.split(' '))
print('map     ->', lines_as_lists.count(), 'elements')
print(lines_as_lists.take(3))

In [ ]:
%%time
# flatMap: each list is "flattened" -> an RDD of individual words
flat_words = rdd.flatMap(lambda line: line.split(' '))
print('flatMap ->', flat_words.count(), 'elements')
print(flat_words.take(10))
print(flat_words.toDebugString().decode())

**Explanation (cell magic `%%time`)**

`%%time` measures the execution time of the whole cell (CPU time and wall time).
Comparing the two cells shows what `flatMap` does:

- `map` applies the function to each line and keeps **one output per input**: we get as
  many elements as lines, and each element is a *list* of words.
- `flatMap` applies the same function but then **flattens** the returned lists: each word
  becomes its own element, so the RDD has many more elements (one per word).

Both are narrow transformations (no shuffle), so the timing mostly measures the `count()`
and `take()` actions that trigger the job. `toDebugString()` shows the lineage Spark
built: the text file RDD followed by the Python `flatMap` step.

In the next cell, `flatMap` produces the words and `.map(lambda word: (word, 1))` turns
each word into a key–value pair `(word, 1)`, the classic starting point of a word count.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [ ]:
# a. count the occurence of each word
from operator import add

# reduceByKey adds the 1s of identical keys (with a local combine before the shuffle)
counts = words.reduceByKey(add)
counts.take(20)

In [ ]:
# b. a common first step in text analysis, change all capital letters to lower case
lower_counts = (rdd
                .flatMap(lambda line: line.lower().split(' '))
                .map(lambda word: (word, 1))
                .reduceByKey(add))

print('distinct words before lower():', counts.count())
print('distinct words after  lower():', lower_counts.count())
lower_counts.take(20)

In [ ]:
# c. eliminate the stop words.
STOP_WORDS_EN = set('''
a about above after again against all am an and any are aren't as at be because
been before being below between both but by can can't cannot could couldn't did
didn't do does doesn't doing don't down during each few for from further had
hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it
it's its itself let's me more most mustn't my myself no nor not of off on once
only or other ought our ours ourselves out over own same shan't she she'd she'll
she's should shouldn't so some such than that that's the their theirs them
themselves then there there's these they they'd they'll they're they've this
those through to too under until up very was wasn't we we'd we'll we're we've
were weren't what what's when when's where where's which while who who's whom
why why's with won't would wouldn't you you'd you'll you're you've your yours
yourself yourselves
s t d ll m re ve o y upon one would said could shall will may might must
mr mrs sir yes also us
'''.split())

no_stop = lower_counts.filter(lambda kv: kv[0] not in STOP_WORDS_EN)
no_stop.take(20)

In [ ]:
# d. sort in alphabetical order
alpha_sorted = no_stop.sortByKey(ascending=True)
alpha_sorted.take(20)

In [ ]:
# e. sort descending by word frequency
freq_sorted = no_stop.sortBy(lambda kv: kv[1], ascending=False)
freq_sorted.take(20)

In [ ]:
# f. remove punctuations and blank spaces
import string

# string.punctuation + typographic characters found in Gutenberg texts
PUNCTUATION = string.punctuation + '“”‘’«»—–…•'
# every punctuation sign is replaced by a space, so "fogg's" -> "fogg s"
PUNCT_TABLE = str.maketrans({c: ' ' for c in PUNCTUATION})

clean_counts = (rdd
                .map(lambda line: line.lower().translate(PUNCT_TABLE))
                # split() with no argument splits on any whitespace
                # and never returns empty strings -> blank spaces removed
                .flatMap(lambda line: line.split())
                .filter(lambda w: w not in STOP_WORDS_EN)
                .map(lambda w: (w, 1))
                .reduceByKey(add)
                .sortBy(lambda kv: kv[1], ascending=False))

clean_counts.take(20)

### Final function: all transformations chained together

The Gutenberg files start and end with an English licence header/footer, which is not
part of the novel. The helper below keeps only the lines between the
`*** START OF` and `*** END OF` markers (if they exist).

In [ ]:
def gutenberg_body(lines_rdd):
    """Keep only the text between the Gutenberg START and END markers."""
    indexed = lines_rdd.zipWithIndex()
    start = (indexed.filter(lambda x: x[0].startswith('*** START OF'))
                    .map(lambda x: x[1]).take(1))
    end = (indexed.filter(lambda x: x[0].startswith('*** END OF'))
                  .map(lambda x: x[1]).take(1))
    start = start[0] if start else -1
    end = end[0] if end else float('inf')
    return indexed.filter(lambda x: start < x[1] < end).keys()


def word_count(path, stop_words=STOP_WORDS_EN, sort_by='frequency',
               strip_header=True):
    """Full pipeline: read -> clean -> tokenize -> filter -> count -> sort.

    sort_by: 'frequency' (descending, ties broken alphabetically)
             or 'alpha' (alphabetical order).
    """
    lines = sc.textFile(path)
    if strip_header:
        lines = gutenberg_body(lines)

    counts = (lines
              .map(lambda line: line.lower())                # b. lower case
              .map(lambda line: line.translate(PUNCT_TABLE)) # f. punctuation
              .flatMap(lambda line: line.split())            # f. blanks
              .filter(lambda w: w not in stop_words)         # c. stop words
              .map(lambda w: (w, 1))
              .reduceByKey(add))                             # a. count

    if sort_by == 'alpha':
        return counts.sortByKey()                            # d. alphabetical
    return counts.sortBy(lambda kv: (-kv[1], kv[0]))         # e. frequency


result = word_count('around_the_world_in_80_days.txt')
result.take(25)

In [ ]:
word_count('around_the_world_in_80_days.txt', sort_by='alpha').take(25)

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
                          ("TD", 35), ("Brooke", 25)])


agesRDD = (dataRDD
  .map(lambda x: (x[0], (x[1], 1)))
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

agesRDD.collect()

**Summary:** this block computes the **average age per name**.
Brooke appears twice (20 and 25), so after `reduceByKey` she becomes
`("Brooke", (45, 2))` and the last `map` gives `45 / 2 = 22.5`.
The other names appear once, so their average is simply their age.
Expected result (order may vary between runs because of the shuffle):
`[('Brooke', 22.5), ('Denny', 31.0), ('Jules', 30.0), ('TD', 35.0)]`.

Carrying the pair `(sum, count)` through `reduceByKey` is the standard way to compute an
average in Spark: the function is associative and commutative, so Spark can pre-aggregate
on each partition before the shuffle (unlike `groupByKey`).

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [ ]:
import time
import statistics


def timer(build_rdd, n_runs=5):
    """Time an RDD pipeline end to end.

    build_rdd: function returning a (lazy) RDD. The RDD is rebuilt at
    each run and an action (collect) forces execution. Returns the result
    and prints min / median wall time over n_runs.
    """
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        result = build_rdd().collect()
        times.append(time.perf_counter() - start)
    print(f'{build_rdd.__name__:<12} min = {min(times):.3f}s   '
          f'median = {statistics.median(times):.3f}s   ({n_runs} runs)')
    return result, times

In [ ]:
PATH = 'around_the_world_in_80_days.txt'


def clean_tokens(word):
    """Lower case + punctuation removal on one raw token."""
    return word.lower().translate(PUNCT_TABLE).split()


def naive():
    return (sc.textFile(PATH)
            .flatMap(lambda line: line.split(' '))
            .map(lambda w: (w, 1))
            .sortByKey()
            .flatMap(lambda kv: [(t, kv[1]) for t in clean_tokens(kv[0])])
            .groupByKey()
            .mapValues(sum)
            .filter(lambda kv: kv[0] not in STOP_WORDS_EN)
            .sortBy(lambda kv: (-kv[1], kv[0])))


def optimized():
    # Cheap narrow transformations first, one aggregation, one final sort
    return (sc.textFile(PATH)
            .map(lambda line: line.lower().translate(PUNCT_TABLE))
            .flatMap(lambda line: line.split())
            # filter as early as possible -> less data everywhere after
            .filter(lambda w: w not in STOP_WORDS_EN)
            .map(lambda w: (w, 1))
            # reduceByKey combines locally before shuffling
            .reduceByKey(add)
            # the sort now runs on a few thousand distinct words only
            .sortBy(lambda kv: (-kv[1], kv[0])))


res_naive, t_naive = timer(naive)
res_opt, t_opt = timer(optimized)

print('\nsame result:', res_naive == res_opt)
print(f'speed-up (median): x{statistics.median(t_naive) / statistics.median(t_opt):.2f}')

**Explanation of the optimisation**

Both pipelines return exactly the same word counts, only the order of the steps changes:

1. **Clean and filter first.** Lower-casing, punctuation removal and stop-word filtering
   are *narrow* transformations (no data exchange between partitions). Doing them first
   shrinks the data: stop words alone represent roughly half of the tokens of an English
   text, so every later step handles far fewer records.
2. **`reduceByKey` instead of `groupByKey`.** `reduceByKey` pre-aggregates on each
   partition (map-side combine) and only sends one `(word, partial_count)` per word and
   partition through the shuffle. `groupByKey` sends every single `(word, 1)`.
3. **Sort only once, at the end.** Sorting is a wide transformation (shuffle + range
   partitioning). In the naive version we sort all raw words; in the optimised one we sort
   only the distinct words after aggregation.
4. **Fewer shuffles overall:** the naive version has 3 shuffles on big data
   (`sortByKey`, `groupByKey`, `sortBy`), the optimised one has 2, on much smaller data.

On a single book in local mode, part of the time is fixed Spark overhead (job scheduling,
Python workers), so the speed-up is moderate; that is also why we take the min/median of
several runs. On a real cluster with a large corpus, the gap grows with the data volume,
because the shuffle (network + disk) becomes the dominant cost.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [ ]:
STOP_WORDS_FR = set('''
a à ai aie aient aies ait alors as au aucun aucune aupres auquel aura aurai
auraient aurais aurait auras aurez auriez aurions aurons auront aussi autre
autres aux auxquelles auxquels avaient avais avait avant avec avez aviez avions
avoir avons ayant ayez ayons bon c ça car ce ceci cela celle celles celui cependant
ces cet cette ceux chaque ci comme comment d dans de dehors déjà depuis des
desquelles desquels deux devait doit donc dont du elle elles en encore entre es
est et étaient étais était étant été êtes étiez étions être eu eue eues eûmes
eurent eus eut eux fait faire fois font furent fus fut hors ici il ils j je
jusqu jusque l la là laquelle le lequel les lesquelles lesquels leur leurs lui
m ma mais me même mêmes mes moi mon n ne ni nos notre nous on ont ou où par
parce pas peu peut plus pour pourquoi qu quand que quel quelle quelles quels
qui quoi s sa sans se sera serai seraient serait seras serez seriez serions
serons seront ses si son sont sous soyez soyons suis sur t ta te tes toi ton
tous tout toute toutes très tu un une vers voici voilà vos votre vous y
dit monsieur mr m
'''.split())

EN, FR = 'around_the_world_in_80_days.txt', 'le_tour_du_monde_en_80_jours.txt'

en_counts = word_count(EN, stop_words=STOP_WORDS_EN)
fr_counts = word_count(FR, stop_words=STOP_WORDS_FR)
en_counts.take(15), fr_counts.take(15)

In [ ]:
import re


def text_stats(path, counts_rdd):
    """Basic EDA figures for one book (Gutenberg header/footer removed)."""
    lines = gutenberg_body(sc.textFile(path)).cache()
    tokens = (lines
              .map(lambda l: l.lower().translate(PUNCT_TABLE))
              .flatMap(lambda l: l.split())
              .cache())
    n_tokens = tokens.count()
    vocab = tokens.distinct().count()
    text = ' '.join(lines.collect())
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    stats = {
        'lines': lines.count(),
        'non-empty lines': lines.filter(lambda l: l.strip()).count(),
        'characters': lines.map(len).sum(),
        'tokens (all words)': n_tokens,
        'vocabulary (distinct words)': vocab,
        'type/token ratio': round(vocab / n_tokens, 4),
        'avg word length': round(tokens.map(len).mean(), 2),
        'sentences (approx.)': len(sentences),
        'avg words / sentence': round(n_tokens / len(sentences), 1),
        'hapax (words seen once)':
            tokens.map(lambda w: (w, 1)).reduceByKey(add)
                  .filter(lambda kv: kv[1] == 1).count(),
        'content words (no stop words)': counts_rdd.map(lambda kv: kv[1]).sum(),
        'chapters':
            lines.filter(lambda l: re.match(r'^\s*(CHAPTER|CHAPITRE)\b', l))
                 .count(),
    }
    lines.unpersist()
    tokens.unpersist()
    return stats


import pandas as pd

stats = pd.DataFrame({
    'English': text_stats(EN, en_counts),
    'French (original)': text_stats(FR, fr_counts),
})
stats['FR / EN'] = (stats['French (original)'] / stats['English']).round(2)
stats

In [ ]:

top_n = 20
top_en = pd.DataFrame(en_counts.take(top_n), columns=['EN word', 'EN count'])
top_fr = pd.DataFrame(fr_counts.take(top_n), columns=['FR word', 'FR count'])
top = pd.concat([top_en, top_fr], axis=1)
top.index = range(1, len(top) + 1)
top

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (title, rdd_counts, color) in zip(
        axes, [('English', en_counts, 'tab:blue'),
               ('French (original)', fr_counts, 'tab:red')]):
    words_, freqs = zip(*rdd_counts.take(15))
    ax.barh(words_[::-1], freqs[::-1], color=color)
    ax.set_title(f'{title} - top 15 words (no stop words)')
    ax.set_xlabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# Main characters and key words: how do they appear in both versions?
en_dict = dict(en_counts.collect())
fr_dict = dict(fr_counts.collect())

pairs = [('fogg', 'fogg'), ('passepartout', 'passepartout'), ('fix', 'fix'),
         ('aouda', 'aouda'), ('phileas', 'phileas'), ('master', 'maître'),
         ('time', 'temps'), ('days', 'jours'), ('train', 'train'),
         ('steamer', 'paquebot'), ('london', 'londres'),
         ('detective', 'détective'), ('elephant', 'éléphant')]

chars = pd.DataFrame(
    [(en, en_dict.get(en, 0), fr, fr_dict.get(fr, 0)) for en, fr in pairs],
    columns=['EN word', 'EN count', 'FR word', 'FR count'])
chars

In [ ]:
# Word length distribution in both languages
def length_distribution(path):
    return dict(gutenberg_body(sc.textFile(path))
                .map(lambda l: l.lower().translate(PUNCT_TABLE))
                .flatMap(lambda l: l.split())
                .map(lambda w: (min(len(w), 15), 1))
                .reduceByKey(add)
                .collect())

en_len, fr_len = length_distribution(EN), length_distribution(FR)
lengths = pd.DataFrame({'English': en_len, 'French': fr_len}).sort_index()
(lengths / lengths.sum()).plot.bar(figsize=(10, 4), color=['tab:blue', 'tab:red'])
plt.title('Word length distribution (share of tokens, 15 = 15+)')
plt.xlabel('word length')
plt.ylabel('share')
plt.show()

### Comparison – observations

*(The numbers come from the tables above; re-read them after running the notebook.)*

- **Same story, same main entities.** Once stop words are removed, both top lists are
  dominated by the characters: *Fogg*, *Passepartout*, *Fix*, *Aouda*, *Phileas*. Proper
  nouns are not translated, so their counts are very close in both versions; small gaps
  come from the translator sometimes replacing a name by a pronoun or a title
  (*"his master"* / *"son maître"*, *"the detective"* / *"le détective"*).
- **Length of the text.** The French original and the English translation have a similar
  number of chapters (37), but not the same number of tokens: French uses more function
  words (articles, contractions like *l'*, *d'*, *qu'* that our tokenizer splits into
  separate tokens), which inflates its token count.
- **Vocabulary richness.** French has a richer morphology (gender and number agreement,
  many verb conjugations: *dit, disait, dirent...*), so the same idea gives more distinct
  word forms. The vocabulary size and the number of hapax (words seen once) are therefore
  typically higher in French, even with a similar story length.
- **Word length.** French words are on average slightly longer, and the length
  distribution is shifted to the right (fewer 1–3 letter words once elisions are removed,
  more long words ending in *-ment*, *-tion*...).
- **Stop words matter.** Each language needs its own stop-word list: applying the English
  list to the French text would leave *de, la, le, et...* at the top of the ranking.
- **Limits of this EDA.** Counts are on surface forms: without lemmatisation
  (*jour / jours*, *day / days*) or stemming, related words are counted separately, and
  a translation is not word-to-word, so only rough comparisons make sense (proportions,
  ratios, rankings) rather than exact counts.

In [ ]:
# stop the Spark session at the end of the lab
spark.stop()